In [1]:
def print_relationships(relationships):

    print("\nRELATIONSHIPS")
    print("=" * 80)

    for relationship in relationships:

        left = (
            f"{relationship['left_table']}."
            f"{relationship['left_column']}"
        )

        right = (
            f"{relationship['right_table']}."
            f"{relationship['right_column']}"
        )

        join_type = relationship.get(
            "join_type",
            "INNER"
        ).upper()

        operator = {
            "eq": "=",
            "neq": "!=",
            "gt": ">",
            "gte": ">=",
            "lt": "<",
            "lte": "<="
        }.get(
            relationship.get("operator"),
            relationship.get("operator", "?")
        )

        print(
            f"{left:<35} "
            f"--[{join_type} JOIN]--> "
            f"{right:<35} "
            f"({operator})"
        )

In [2]:
# import os
# import json

# json_path = 'output\\data.json'

# if not os.path.exists(json_path):
#     with open(json_path, "w") as f:
#         json.dump({}, f, indent=4)
        
# with open(json_path) as f:
#         data = json.load(f)

# print(data)

# for i in data.keys():
#     print(i)
    
# existing = [
#     key for key in data.keys()
# ]

# existing

In [3]:
def save(file_path, relationships):

    with open(
        file_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            relationships,
            file,
            ensure_ascii=False,
            indent=4
        )

In [4]:
import json
from dataclasses import asdict
from extractor import SemanticExtractor

sql = """
with main as (
    select
        m.id,
        m.dep_id
    from
        cbs.main as m
    union all
    select
        c.id,
        c.dep_id
    from
        cbs.other as c
)
,table_1 as (
    select
        user.id
    from
        CBS.C_USER as user
    inner join
        (
            select
                Distinct on (id, dep_id)
                m3.id,
                m3.dep_id
            from
                main as m3
            order by
                id,
                dep_id
        ) as m
        on m.id = user.main_id
        and m.dep_id = user.dep_id
)
select
    e.id,
    e.dep_id
from
    cbs.e_reqexc as e
left join
    table_1 t1
    on t1.id = e.tus_id
right join
    (
        select
            i.id,
            cds.dep_id
        from
            cbs.i_dea as i
        inner join
            cbs.c_dep_std as cds
            on cds.dep_id = i.dea_dep_id
        where
            rn = 1
    ) as m2
    on m2.id = e.id
    and m2.dep_id = e.dep_id
union all
select
    t.id,
    t.dep_id
from
    cbs.t_dea as t
inner join
    main as m
    on m.id = t.id
    and m.dep_id = t.dep_id
right join
    (
        select
            i.id,
            cds.dep_id
        from
            cbs.i_dea as i
        inner join
            cbs.c_dep_std as cds
            on cds.dep_id = i.dea_dep_id
        where
            rn = 1
    ) as m2
    on m2.id = t.id
    and m2.dep_id = t.dep_id
union
select
    t2.id,
    t2.dep_id
from
    cbs.t_dea as t2
inner join
    main as m2
    on m2.id = t2.id
    and m2.dep_id = t2.dep_id
right join
    (
        select
            i.id,
            cds.dep_id
        from
            cbs.i_dea as i
        inner join
            cbs.c_dep_std as cds
            on cds.dep_id = i.dea_dep_id
        where
            rn = 1
    ) as m3
    on m3.id = t2.id
    and m3.dep_id = t2.dep_id
"""

relationships_print, relationships_json = SemanticExtractor().extract(sql)

print_relationships(relationships_print)

file_path = "..\\code\\output\\data.json"
save(file_path, relationships_json)


RELATIONSHIPS
cbs.c_user.id                       --[LEFT JOIN]--> cbs.e_reqexc.tus_id                 (=)
cbs.e_reqexc.id                     --[RIGHT JOIN]--> cbs.i_dea.id                        (=)
cbs.c_dep_std.dep_id                --[RIGHT JOIN]--> cbs.e_reqexc.dep_id                 (=)
cbs.main.id                         --[INNER JOIN]--> cbs.t_dea.id                        (=)
cbs.other.id                        --[INNER JOIN]--> cbs.t_dea.id                        (=)
cbs.main.dep_id                     --[INNER JOIN]--> cbs.t_dea.dep_id                    (=)
cbs.other.dep_id                    --[INNER JOIN]--> cbs.t_dea.dep_id                    (=)
cbs.i_dea.id                        --[RIGHT JOIN]--> cbs.t_dea.id                        (=)
cbs.c_dep_std.dep_id                --[RIGHT JOIN]--> cbs.t_dea.dep_id                    (=)
cbs.c_user.main_id                  --[INNER JOIN]--> cbs.main.id                         (=)
cbs.c_user.main_id                  --[INNER J

In [5]:
from pathlib import Path
import traceback
import pandas as pd

SQL_FOLDER = '..\\code\\scripts'
JSON_SAVE_PATH = 'output\\data.json'
CSV_SAVE_PATH = 'output\\results.csv'

results = []

folder = Path(SQL_FOLDER)
files = list(folder.glob("*.sql"))


for i, file in enumerate(files, start=1):
    try:
        print(f"\n[{i}/{len(files)}] {file.name}", end=" ")
        
        with open(file, 'r', encoding='utf-8') as f:
            query = f.read()
            
        relationships_print, relationships_json = SemanticExtractor().extract(sql=query, save_path=JSON_SAVE_PATH)
        
        results.append(
            {
                "file_name": file.name,
                "status": "Done",
                "error_type": None,
                "error": None,
                "fulltext": None
            }
        )
        print("✅ Done")
        
        print_relationships(relationships_print)
        
    except Exception as e:
        results.append(
            {
                "file_name": file.name,
                "status": "Error",
                "error_type": type(e).__name__,
                "error": str(e),
                "fulltext": traceback.format_exc()
            }
        )
        print(f"❌ {type(e).__name__}: {e}")
        # print(f"Full error text: {traceback.format_exc()}")

df = pd.DataFrame(results)
df.to_csv(CSV_SAVE_PATH)

print(f"\nОбработано: {len(files)}")
print(f"Успешно: {sum(r['status'] == 'Done' for r in results)}")
print(f"Ошибок: {sum(r['status'] == 'Error' for r in results)}")


[1/2] scripts_2.sql ✅ Done

RELATIONSHIPS
cbs.c_user.main_id                  --[INNER JOIN]--> cbs.main.id                         (=)
cbs.c_user.main_id                  --[INNER JOIN]--> cbs.other.id                        (=)
cbs.c_user.dep_id                   --[INNER JOIN]--> cbs.main.dep_id                     (=)
cbs.c_user.dep_id                   --[INNER JOIN]--> cbs.other.dep_id                    (=)

[2/2] scripts_4.sql ✅ Done

RELATIONSHIPS
cbs.c_user.main_id                  --[INNER JOIN]--> cbs.table_1.id                      (=)
cbs.c_user.main_id                  --[INNER JOIN]--> cbs.other.id                        (=)
cbs.c_user.dep_id                   --[INNER JOIN]--> cbs.table_1.dep_id                  (=)
cbs.c_user.dep_id                   --[INNER JOIN]--> cbs.other.dep_id                    (=)
cbs.c_dep_user.ord_id               --[INNER JOIN]--> cbs.table_1.id                      (=)
cbs.c_dep_user.ord_id               --[INNER JOIN]--> cbs.other.id  